# P2C-64 v3.2 (`WACA_G2_V32` ＋ 双方公平 `B4_HYBRID` Move Ordering) — Google Colab 実行ノートブック

Egaroucid 公開学習データ（約2,551万局面）および既存の v2.0 Phase-1 ベースライン（`raw_ntuple64.p2c`）を再利用し、**P2C-64 v3.2（`WACA_G2_V32`）** を学習・ビット完全一致監査・純スカラー実戦対局（20ms / 50ms / 100ms・`--mo static` vs `--mo b4_hybrid`・`tt_guard 5.5 vs 6.5`・`v3.2 vs v3.1`・MPC $\sigma$）するためのワンクリックノートブックです。

### v3.2 の 3 大革新点
1. **Stage 3 純スカラー高速化（融合 16 ロード ＋ 28 次元精選 ＋ 4-way `j-tile` MLP）**：v3.1 の 40 ロードを `p=0..13` 融合＋`p=62..63` の 16 ロード（`-60%`）へ圧縮し、連続配列 `f[28]` の 4-way `j-tile`（`f[i]` ロード 1/4）により Stage 3 レイテンシを大幅短縮（0 SIMD・0 PLT 呼び出し）。
2. **`ply 19..25` 滑らか線形ランプ付き 4 近傍空きパリティ**：v3.1 の `ply=24` ハード切替を `ramp = clamp(ply - 18, 0, 8)` の厳密整数ランプへ刷新し、中盤 `d4→d8` MPC $\sigma$ の安定性を改善（学習平滑化デフォルト `1.2e-3 / 6.0e-3` ＋ QAT 劣化ゼロ機構維持）。
3. **双方 100% 公平な `B4_HYBRID` Move Ordering 統合**：`depth >= 4` の Fastest-First ＋ `depth == 2..3` の角占有文脈・2スロット Killer・History を両エンジンに完全同一条件で適用可能（`--mo static|b4_hybrid`）。

- **既存成果物を完全保護**：既存 `cache_bin/`・`run_egaroucid_25m_v20/`・`run_egaroucid_25m_v30/`・`run_egaroucid_25m_v31/` は読み取り再利用のみ。v3.2 の全成果物は新規ディレクトリ `run_egaroucid_25m_v32/` へ保存します。

## Cell 1: GPU/VRAM 自動判定 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v32"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
V30_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v30"
V31_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v31"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (recommended for Cell 6A / Cell 6B pure-scalar benchmarks & MPC sigma)")

TRAIN_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
MLP_HIDDEN = 16   # Default fast pure-scalar H=16
TRAIN_H24 = False # Set True to also train optional H=24 variant in Cell 4B
print(f"Config  : RUN_NAME={RUN_NAME}, TRAIN_BATCH={TRAIN_BATCH}, MLP_HIDDEN={MLP_HIDDEN}, TRAIN_H24={TRAIN_H24}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ 純スカラービルド・単体テスト・27カーネル 0-SIMD ASM監査

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: 既存キャッシュ `train.bin` / `val.bin` および凍結 Phase-1 `raw_ntuple64.p2c` の再利用準備

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

for src, dst in [(cache_tr, local_tr), (cache_va, local_va)]:
    if os.path.exists(dst):
        print("[OK] Local SSD cache present:", dst)
    else:
        assert os.path.exists(src), f"Required binary cache not found on Drive: {src}"
        print(f"[Cache Hit] Copying {src} -> {dst} ...")
        shutil.copy2(src, dst)

# Reuse frozen Phase-1 baseline from run_egaroucid_25m_v20 (0-min Phase 1!)
v20_raw = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
v32_raw = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
local_raw = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"

if os.path.exists(v32_raw):
    print("[OK] Reusing raw_ntuple64.p2c in v3.2 work dir:", v32_raw)
elif os.path.exists(v20_raw):
    print("[Reuse v2.0 Phase-1] Copying frozen raw_ntuple64.p2c -> v3.2 work dir (0-min Phase 1!)...")
    shutil.copy2(v20_raw, v32_raw)
else:
    print("[Fallback] v2.0 raw_ntuple64.p2c not found; running Phase 1 once...")
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_p2c64.py",
        "--train-bin", local_tr,
        "--val-bin", local_va,
        "--work-dir", DRIVE_WORK_DIR,
        "--phases", "phase1",
        "--resume",
        "--batch-size", str(TRAIN_BATCH),
        "--p1-epochs", "6",
    ], check=True)

shutil.copy2(v32_raw, local_raw)
!ls -lh /content/data/

## Cell 4: WACA-G2 v3.2 本学習（`H=16`: Slim 8-Dim Stage 2.5 → 凍結 → 30-Block EO Key-14 + 28→16→1 MLP + 修正版 QAT）

In [ ]:
!python3 {REPO_DIR}/python/train_waca_g2_v32.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --raw-p2c {LOCAL_DATA_DIR}/raw_ntuple64.p2c \
    --work-dir {DRIVE_WORK_DIR} \
    --out-name waca_g2_v32.p2c \
    --mlp-hidden {MLP_HIDDEN} \
    --batch-size {TRAIN_BATCH} \
    --s25-epochs 3 \
    --s25-lr 2e-2 \
    --s25-smooth 1e-3 \
    --s3-epochs 7 \
    --s3-qat-epochs 2 \
    --qat-lr-mult 0.12 \
    --qat-lr-min 0.02 \
    --s3-emb-lr 8e-3 \
    --s3-mlp-lr 4e-3 \
    --s3-ply-smooth 1.2e-3 \
    --s3-blk-smooth 6.0e-3 \
    --resume

print("\nSaved v3.2 artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics_v32.*

## Cell 4B: （オプション）`H=24` バリアント追加学習（`TRAIN_H24 = True` のときのみ実行・Stage 2.5 は自動再利用）

In [ ]:
import subprocess

if TRAIN_H24:
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_waca_g2_v32.py",
        "--train-bin", f"{LOCAL_DATA_DIR}/train.bin",
        "--val-bin", f"{LOCAL_DATA_DIR}/val.bin",
        "--raw-p2c", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c",
        "--work-dir", DRIVE_WORK_DIR,
        "--out-name", "waca_g2_v32_h24.p2c",
        "--mlp-hidden", "24",
        "--batch-size", str(TRAIN_BATCH),
        "--s25-epochs", "3",
        "--s3-epochs", "7",
        "--s3-qat-epochs", "2",
        "--qat-lr-mult", "0.12",
        "--qat-lr-min", "0.02",
        "--s3-ply-smooth", "1.2e-3",
        "--s3-blk-smooth", "6.0e-3",
        "--resume",
    ], check=True)
else:
    print("[Skip] TRAIN_H24 is False. Set TRAIN_H24 = True in Cell 1 if you want to also train waca_g2_v32_h24.p2c.")

## Cell 5: C++/Python 全3段ビット完全一致（`S1=0 S2.5=0 S3=0`）＆ 4世代後方互換・16対称（$D_4 \times \mathbb{Z}_2$）監査

In [ ]:
import os
import subprocess

models = [f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c", f"{DRIVE_WORK_DIR}/waca_g2_v32.p2c"]
for cand in [
    f"{DRIVE_WORK_DIR}/waca_g2_v32_h24.p2c",
    f"{V31_RUN_DIR}/waca_g2_v31.p2c",
    f"{V30_RUN_DIR}/waca_g2.p2c",
]:
    if os.path.exists(cand):
        models.append(cand)

val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"
subprocess.run([
    "python3", f"{REPO_DIR}/python/test_bit_exact_io.py",
    "--data", val_bin,
    "--num-pos", "1000",
    "--models", *models,
    "--bench-bridge",
], check=True)

## Cell 6A: （CPUランタイム推奨・前半）MPC $\sigma$ ＋ 20ms 対局（`--mo static` / `--mo b4_hybrid` / `v3.2 vs v3.1` / MOアブレーション）

GPUを切断し CPU ランタイムへ切り替えた後、**Cell 1 → Cell 2 → Cell 6A** を実行してください。各対局の完了直後に CSV 最終行を `print` します：
1. **MPC $\sigma$（`d2→d6`, `d4→d8`, 2,000局面）**：`RAW_NTUPLE_64` vs `WACA_G2_V31`（存在時） vs `WACA_G2_V32`
2. **MOアブレーション**：`RAW(b4_hybrid) vs RAW(static)`（20ms, 400局）
3. **`v3.2 vs RAW`（20ms, 1,000局, `--mo static`, `wide_safe_bnd 2.5/1.5 tt_guard=5.5`）**
4. **`v3.2 vs RAW`（20ms, 1,000局, `--mo b4_hybrid`, `wide_safe_bnd 2.5/1.5 tt_guard=5.5`）**
5. **`v3.2 vs RAW`（20ms, 1,000局, `--mo b4_hybrid`, `wide_safe_bnd 3.5/2.1 tt_guard=5.5`）**
6. **`v3.2 vs v3.1` 直接対決（20ms, 1,000局, `--mo static`, `wide_safe_bnd 2.5/1.5 tt_guard=5.5`, v3.1存在時自動実行）**

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V32_P2C = f"{DRIVE_WORK_DIR}/waca_g2_v32.p2c"
V31_P2C = f"{V31_RUN_DIR}/waca_g2_v31.p2c"
CSV_V32 = f"{DRIVE_WORK_DIR}/match_results_v32.csv"
val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

assert os.path.exists(RAW_P2C), f"Missing RAW model: {RAW_P2C}"
assert os.path.exists(V32_P2C), f"Missing v3.2 model: {V32_P2C}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

# 1) MPC sigma measurement (d2->d6 and d4->d8, 2,000 pos)
mpc_models = [RAW_P2C]
if os.path.exists(V31_P2C):
    mpc_models.append(V31_P2C)
mpc_models.append(V32_P2C)

for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== P2C-64 v3.2 MPC Sigma: depth {sh} -> {dp} (2,000 pos) ===", flush=True)
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5",
        "--m3", "1.5",
        "--tt-guard", "5.5",
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_v32_d{sh}_d{dp}.csv",
        "--models", *mpc_models,
    ], check=True)

# 2) MO Ablation: RAW(b4_hybrid) vs RAW(static) (20ms, 400 games)
run_match_and_log(
    "[6A-1] MO Ablation: RAW(b4_hybrid) vs RAW(static) (20ms, 400g)",
    [
        MATCH_BIN, "--p1", RAW_P2C, "--p2", RAW_P2C,
        "--mode", "time", "--limit", "20", "--pairs", "200",
        "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
        "--mo-p1", "b4_hybrid", "--mo-p2", "static",
        "--seed", "20261126", "--csv", CSV_V32,
    ],
    CSV_V32,
)

# 3) 20ms Fair Matches: WACA_G2_V32 vs RAW_NTUPLE_64
suite_6a = [
    ("[6A-2] 20ms STATIC (wide_safe_bnd 2.5/1.5 g=5.5, 1000g)",    "static",    "2.5", "1.5", "5.5"),
    ("[6A-3] 20ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g=5.5, 1000g)", "b4_hybrid", "2.5", "1.5", "5.5"),
    ("[6A-4] 20ms B4_HYBRID (wide_safe_bnd 3.5/2.1 g=5.5, 1000g)", "b4_hybrid", "3.5", "2.1", "5.5"),
]
for label, mo, m25, m3, ttg in suite_6a:
    run_match_and_log(
        f"{label}: WACA_G2_V32 vs RAW_NTUPLE_64",
        [
            MATCH_BIN, "--p1", V32_P2C, "--p2", RAW_P2C,
            "--mode", "time", "--limit", "20", "--pairs", "500",
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", mo, "--gate-mode", "wide_safe_bnd",
            "--m25", m25, "--m3", m3, "--tt-guard", ttg,
            "--seed", "20261126", "--csv", CSV_V32,
        ],
        CSV_V32,
    )

# 4) Head-to-head v3.2 vs v3.1 (if v3.1 exists on Drive)
if os.path.exists(V31_P2C):
    run_match_and_log(
        "[6A-5] Head-to-Head 20ms STATIC (wide_safe_bnd 2.5/1.5 g=5.5, 1000g): WACA_G2_V32 vs WACA_G2_V31",
        [
            MATCH_BIN, "--p1", V32_P2C, "--p2", V31_P2C,
            "--mode", "time", "--limit", "20", "--pairs", "500",
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", "static", "--gate-mode", "wide_safe_bnd",
            "--m25", "2.5", "--m3", "1.5", "--tt-guard", "5.5",
            "--seed", "20261126", "--csv", CSV_V32,
        ],
        CSV_V32,
    )

print(f"\n[Cell 6A Done] Results saved to: {CSV_V32}")

## Cell 6B: （CPUランタイム推奨・後半）超深読み 50ms / 100ms 検証（`--mo b4_hybrid` vs `--mo static` ＆ `tt_guard 5.5` vs `6.5`）

セッション切断に強い後半専用セルです。50ms（各 600 局）および 100ms（各 500 局）の超深読み帯で、`--mo b4_hybrid` / `--mo static` と `tt_guard`（`5.5` vs `6.5`）を完全公平条件で検証し、毎試合ごとに CSV 最終行を表示します。

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V32_P2C = f"{DRIVE_WORK_DIR}/waca_g2_v32.p2c"
CSV_V32 = f"{DRIVE_WORK_DIR}/match_results_v32.csv"

assert os.path.exists(RAW_P2C), f"Missing RAW model: {RAW_P2C}"
assert os.path.exists(V32_P2C), f"Missing v3.2 model: {V32_P2C}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

suite_6b = [
    ("[6B-1] 50ms  B4_HYBRID (wide_safe_bnd 2.5/1.5 g=5.5, 600g)", "50",  "300", "b4_hybrid", "2.5", "1.5", "5.5"),
    ("[6B-2] 50ms  B4_HYBRID (wide_safe_bnd 2.5/1.5 g=6.5, 600g)", "50",  "300", "b4_hybrid", "2.5", "1.5", "6.5"),
    ("[6B-3] 50ms  STATIC    (wide_safe_bnd 2.5/1.5 g=5.5, 600g)", "50",  "300", "static",    "2.5", "1.5", "5.5"),
    ("[6B-4] 100ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g=5.5, 500g)", "100", "250", "b4_hybrid", "2.5", "1.5", "5.5"),
    ("[6B-5] 100ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g=6.5, 500g)", "100", "250", "b4_hybrid", "2.5", "1.5", "6.5"),
    ("[6B-6] 100ms STATIC    (wide_safe_bnd 2.5/1.5 g=5.5, 500g)", "100", "250", "static",    "2.5", "1.5", "5.5"),
]

for label, limit, pairs, mo, m25, m3, ttg in suite_6b:
    run_match_and_log(
        f"{label}: WACA_G2_V32 vs RAW_NTUPLE_64",
        [
            MATCH_BIN, "--p1", V32_P2C, "--p2", RAW_P2C,
            "--mode", "time", "--limit", limit, "--pairs", pairs,
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", mo, "--gate-mode", "wide_safe_bnd",
            "--m25", m25, "--m3", m3, "--tt-guard", ttg,
            "--seed", "20261126", "--csv", CSV_V32,
        ],
        CSV_V32,
    )

print(f"\n=== [Cell 6B Done] Full CSV Summary ({CSV_V32}) ===")
!cat {CSV_V32}